PII scrubing

To build a PII (Personally Identifiable Information) scrubbing application, combine field-level rules for structured data with pattern matching and NLP/ML models for unstructured text. You can explore a structured extraction example in the [BAML PII Data Extraction and Scrubbing guide](https://docs.boundaryml.com/examples/prompt-engineering/pii-data-extraction-scrubbing) or an LLM gateway approach in the [Hugging Face LLM Gateway for PII Detection guide](https://huggingface.co/learn/cookbook/en/llm_gateway_pii_detection). [1, 2, 3, 4, 5] 
## 1. Define Your Scope and Data Types

* Structured data (JSON, SQL, CSV): Scrub known sensitive keys or columns directly (e.g., email, phone, ssn, address).
* Unstructured text (logs, support tickets, chats): Detect entities like names, emails, phone numbers, IDs, and financial numbers in free text.
* Documents and images (PDFs, PNGs, JPGs): Pair OCR with entity detection and visual redaction. [1, 2, 4, 6, 7, 8, 9, 10] 

## 2. Choose Your Detection and Scrubbing Stack

* Deterministic rules & regex: Fast and predictable for structured fields and fixed patterns (emails, phone numbers, card-like digits). Use as a fast baseline, knowing regex can miss edge cases in free text. [1, 3] 
* NLP & PII libraries: Tools like Microsoft Presidio (Analyzer + Anonymizer) or Python's Scrubadub detect entities with confidence scores and replace or mask them. [2, 6] 
* LLM or structured extraction: Frameworks like BAML or Instructor can extract PII into typed schemas and replace values with placeholders. Run local or self-hosted models when raw data cannot leave your environment. [4, 5, 11, 12] 
* Managed APIs: Cloud services (such as Amazon Comprehend or Azure AI Language PII detection) reduce maintenance if sending data to external APIs fits your compliance rules. [8, 9] 

## 3. Pick a Scrubbing Strategy

* Redaction / masking: Replace values with asterisks (***), [REDACTED], or category tags (<EMAIL_0>). [2, 7, 11] 
* Synthetic replacement / pseudonymization: Swap values with realistic fake data (e.g., via Faker) or deterministic tokens when downstream testing or analytics need realistic formats. [2] 
* Vaulted tokenization (reversible): Store a placeholder-to-original mapping or use a privacy vault only when your workflow requires restoring original values under strict access controls. [4, 13] 

## 4. Build the Application Pipeline

   1. Ingest & parse: Accept API requests, batch files, or stream events, and separate known sensitive fields from free-text payloads.
   2. Detect entities: Run field rules, compiled regexes, and NER/ML detectors to collect entity spans, types, and confidence scores.
   3. Transform values: Apply your chosen masking, placeholder, or synthetic replacement rules without mutating source records unexpectedly.
   4. Validate & log: Check for missed high-risk patterns, and record non-sensitive audit metadata (timestamps, entity counts, policy version) rather than raw PII. [1, 2, 7, 8, 11, 14] 

## 5. Harden for Production

* Scrub at ingress and egress: Clean data before storage, logging, analytics, or external model calls.
* Keep hot paths fast: Avoid heavy ML calls on high-throughput logs or spans unless budgeted for latency.
* Enforce access controls: Treat any token mappings or raw inputs as high-sensitivity data with encryption and role-based access. [1, 3, 8, 13, 14] 

To tailor this into a concrete architecture or starter code, share:

* Your data format and volume (e.g., JSON APIs, CSV batches, logs, or PDFs/images)
* Your preferred language/stack (e.g., Python, Go, Node.js, or cloud services)
* Whether scrubbing needs to be reversible or one-way


[1] [https://langfuse.com](https://langfuse.com/resources/engineering/pii-masking-llm-applications)
[2] [https://www.youtube.com](https://www.youtube.com/watch?v=_dAzcCk-3U4)
[3] [https://www.youtube.com](https://www.youtube.com/watch?v=Ql2gLHWuX7M)
[4] [https://docs.boundaryml.com](https://docs.boundaryml.com/examples/prompt-engineering/pii-data-extraction-scrubbing)
[5] [https://huggingface.co](https://huggingface.co/learn/cookbook/llm_gateway_pii_detection)
[6] [https://dev.socrata.com](https://dev.socrata.com/blog/2017/01/03/scrubbing-data-with-python)
[7] [https://github.com](https://github.com/developertogo/scrub-pii)
[8] [https://aws.amazon.com](https://aws.amazon.com/blogs/machine-learning/redacting-pii-from-application-log-output-with-amazon-comprehend/)
[9] [https://www.youtube.com](https://www.youtube.com/watch?v=d_NVVOW53zY&t=13)
[10] [https://www.reddit.com](https://www.reddit.com/r/SideProject/comments/1velnil/i_built_a_free_opensource_tool_that_scrubs/)
[11] [https://python.useinstructor.com](https://python.useinstructor.com/examples/pii/)
[12] [https://www.youtube.com](https://www.youtube.com/watch?v=2yQ9t7qhjeQ&t=17)
[13] [https://www.youtube.com](https://www.youtube.com/watch?v=a5B61ugbzRI)
[14] [https://medium.com](https://medium.com/@2nick2patel2/data-privacyrag-security-101-guardrails-pii-audits-177cf43ef71b)


In [1]:
%pip install streamlit 

  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.36.2
    Uninstalling protobuf-7.36.2:
      Successfully uninstalled protobuf-7.36.2
Note: you may need to restart the kernel to use updated packages.


In [1]:
!streamlit hello

^C


In [5]:
import streamlit as st
st.button
!streamlit run streamlit.py

Usage: streamlit run [OPTIONS] TARGET [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: streamlit.py


PII masking and security is the practice of finding personal data in text, then replacing or protecting it before it is stored, displayed, logged, or sent to an AI model. For **Service 1 / AI-1**, Microsoft Presidio is a strong default: it combines regex, NER, and context-aware rules to detect PII, then applies masking, redaction, hashing, replacement, or encryption. [microsoft.github](https://microsoft.github.io/presidio/analyzer/index.html)

## What “PII masking” means

PII (Personally Identifiable Information) includes data that can identify a person directly or indirectly: names, emails, phone numbers, addresses, government IDs, payment-card data, IP addresses, medical details, and sometimes account or employee IDs.

**Masking** means making the sensitive portion unusable to a viewer while often retaining limited utility—for example:

| Original | Masked output | Use case |
|---|---|---|
| `john.doe@example.com` | `j***@example.com` | Support can see domain but not full address |
| `4111 1111 1111 1111` | `************1111` | Verify last four digits |
| `+91 98765 43210` | `+91 *****43210` | Partial callback verification |
| `Aadhaar: 1234 5678 9012` | `[AADHAAR_REDACTED]` | No partial value should remain |

Masking is usually **irreversible**. If downstream systems need to recover the original value under strict authorization, use **tokenization** or **encryption** instead; tokenization requires a vault/service to reverse, while encryption is reversible only with the key. [teradata](https://www.teradata.com/insights/data-security/what-is-a-pii-data-privacy-vault)

## Presidio architecture

Presidio has two core components:

- **Analyzer**: detects PII spans and entity types in text using predefined and custom recognizers, regex, NER, and contextual logic. [microsoft.github](https://microsoft.github.io/presidio/analyzer/index.html)
- **Anonymizer**: transforms each detected span using an operator such as `mask`, `redact`, `replace`, `hash`, `encrypt`, or `keep`. [microsoft.github](https://microsoft.github.io/presidio/anonymizer/index.html)

It is a library/SDK rather than a managed SaaS service, so you deploy and operate it inside your own application or infrastructure. [omri374.github](https://omri374.github.io/presidio/faq/)

### Detection methods

| Method | Best for | Example |
|---|---|---|
| Regex | Highly structured values | Email, phone, card number, IP address |
| NER | Contextual natural-language entities | Person, organization, location |
| Checksum/validation | Values with validity rules | Credit-card numbers, some national IDs |
| Custom recognizers | Organization-specific identifiers | `EMP-123456`, patient MRN, internal account ID |
| Context scoring | Reducing false positives | Words such as “email,” “phone,” or “SSN” near a candidate |

Presidio’s built-in recognizers can be extended with custom regex, deny-list, rule-based, or model-based recognizers; this is important for local identifiers and country-specific ID formats. [microsoft.github](https://microsoft.github.io/presidio/analyzer/index.html)

## Recommended AI-1 design

Use a **defense-in-depth pipeline**, not a single regex pass:

1. **Ingress inspection**: Detect PII in user prompts, uploaded documents, chat text, and structured fields.
2. **Normalize text**: Preserve layout where needed, but extract text from PDFs, images, Office files, and HTML before analysis.
3. **Tiered detection**: Run fast regex rules first for high-confidence structured PII, then NER/context analysis for names, locations, and free-text PII. [ehga](https://ehga.org/pii-detection-and-redaction-pipelines-for-llm-inputs-and-outputs)
4. **Policy decision**: Choose an action per entity type, jurisdiction, user role, and data classification.
5. **Anonymize before egress**: Mask or tokenize before sending content to an LLM, third-party API, log store, analytics pipeline, or downstream service.
6. **Output inspection**: Scan model responses too, because a model may echo PII from the prompt, retrieved documents, or prior context.
7. **Audit and monitor**: Log entity type, action, confidence, policy version, and request metadata—never the raw PII.

For LLM use, apply PII masking to both the request and the response. This reduces accidental disclosure through prompts, cached context, logs, and generated answers. [docs.litellm](https://docs.litellm.ai/docs/tutorials/presidio_pii_masking)

## Choosing the right action

| Action | Reversible? | Best use | Caution |
|---|---:|---|---|
| `mask` | No | Display, support workflows, limited verification | Partial values can still be sensitive |
| `redact` | No | Logs, exports, public sharing, strict minimization | Removes usefulness of the value |
| `replace` | Sometimes, if a reversible mapping is maintained | LLM processing with later re-identification | Mapping itself is sensitive |
| `hash` | No for plain hash; pseudonymous with keyed HMAC | Deduplication/analytics without exposing raw value | Use keyed HMAC, not unsalted hashes, for low-entropy PII |
| `encrypt` | Yes with key | Data that must be recovered by authorized systems | Key management becomes critical |
| `tokenize` | Yes via vault | Payments, customer records, cross-system processing | Requires secure token vault and access controls |

Presidio supports masking, replacement, redaction, hashing, and encryption operators; its `mask` operator can replace a chosen number of characters with a configured masking character. [microsoft.github](https://microsoft.github.io/presidio/anonymizer/index.html)

## Example policy

A practical default for an AI service:

| Entity | Detection | Action before LLM | Action in logs |
|---|---|---|---|
| Email | Regex + context | Replace with `[EMAIL_1]` | Redact |
| Phone | Regex + country rules | Mask all but last 2–4 digits | Redact |
| Credit card | Regex + Luhn validation | Redact entirely | Redact |
| Government ID | Regex + validation + context | Redact entirely | Redact |
| Person name | NER + context | Replace with `[PERSON_1]` | Redact |
| Address | NER + address parser | Replace or generalize to city/region | Redact |
| IP address | Regex | Mask or truncate where appropriate | Redact |
| Internal employee/account ID | Custom regex | Tokenize or replace | Redact |

Avoid masking a full Aadhaar, PAN, passport, SSN, or payment-card number with only a few visible characters unless your compliance and fraud teams explicitly approve it. For high-risk identifiers, **redact before the LLM** and retain the original only in an access-controlled vault if re-identification is genuinely required.

## Security controls

PII masking is not sufficient by itself. Pair it with:

- **Data minimization**: Collect and send only fields necessary for the task.
- **Classification**: Label data as public, internal, confidential, restricted, or regulated.
- **Encryption in transit and at rest**: TLS for all service traffic; encrypted storage and backups.
- **Key management**: Use a KMS/HSM, rotate keys, separate encryption keys from application secrets, and restrict decryption permissions.
- **Token vault controls**: Apply least privilege, RBAC/ABAC, short-lived credentials, and approval workflows for de-tokenization. Privacy vaults commonly combine tightly controlled access with request-level audit trails. [teradata](https://www.teradata.com/insights/data-security/what-is-a-pii-data-privacy-vault)
- **No raw PII in logs**: Redact prompts, completions, traces, error messages, URLs, headers, and debugging payloads.
- **Retention limits**: Delete or purge raw prompts, attachments, and derived embeddings according to a documented schedule.
- **Access controls**: Separate roles for PII readers, de-identification operators, administrators, and auditors.
- **Model/data boundaries**: Prevent PII from being sent to unapproved external models, plugins, retrieval stores, or third-party processors.
- **Human review**: Use sampled quality assurance and an escalation path for missed PII or false positives.

## Important limitations

Automated PII detection will produce both **false negatives** (missed PII) and **false positives** (benign text incorrectly flagged). Presidio itself explicitly notes that it cannot guarantee detection of all sensitive information, so additional safeguards are required. [omri374.github](https://omri374.github.io/presidio/faq/)

Common gaps to test:

- Names and addresses written informally or in multiple languages.
- PII split across fields, pages, tables, or OCR errors.
- Indirect identifiers: device IDs, precise geolocation, account numbers, free-text medical details.
- Homoglyphs, spacing, punctuation, and deliberate obfuscation.
- PII introduced by retrieval-augmented generation (RAG) documents or tool outputs.
- Model-generated PII in responses.

## Implementation blueprint

For **Service 1 / AI-1**, a clean reference design is:

```text
User input
   ↓
Schema validation + file/text extraction
   ↓
Presidio Analyzer
   ├─ Regex recognizers
   ├─ NER recognizers
   └─ Custom domain recognizers
   ↓
Policy engine
   ├─ Entity type
   ├─ Risk score
   ├─ User consent / purpose
   └─ Destination (LLM, log, vendor)
   ↓
Presidio Anonymizer
   ├─ Mask / redact / replace / hash / encrypt
   └─ Consistent placeholders
   ↓
LLM / downstream service
   ↓
Output PII scan
   ↓
Redacted audit log + metrics
```

Use stable placeholders such as `[PERSON_1]`, `[EMAIL_1]`, and `[PHONE_1]` when the model needs coherence across a conversation. Maintain the mapping only in a separately secured, access-controlled store, with a documented purpose and retention period.

## Minimal Presidio approach

A typical configuration would:

- Enable entities such as `PERSON`, `EMAIL_ADDRESS`, `PHONE_NUMBER`, `CREDIT_CARD`, `IP_ADDRESS`, `LOCATION`, and relevant national IDs.
- Add custom recognizers for internal identifiers, for example `EMP-[0-9]{6}`.
- Use `replace` for LLM prompts, `redact` for logs, and `mask` only for controlled user-facing displays.
- Run detection with a high threshold for blocking and a lower threshold for human review.
- Continuously evaluate precision and recall on representative production-like samples.

Presidio’s predefined and custom recognizers use regex, NER, and other logic, while its anonymizer applies the selected transformation to detected entities. [microsoft.github](https://microsoft.github.io/presidio/analyzer/index.html)

## Bottom line

For AI-1, use **Presidio as the detection/anonymization engine**, but treat it as one control—not the entire security program. The strongest design is: **detect early, redact before the LLM, scan outputs, keep raw PII out of logs, encrypt/tokenize only when reversibility is required, and tightly control every path that can restore identity.** [teradata](https://www.teradata.com/insights/data-security/what-is-a-pii-data-privacy-vault)

Here is a minimal, production-ready way to set up **Microsoft Presidio in Python** for PII detection and anonymization. Install the analyzer and anonymizer packages, add an NLP model, then send analyzer results to the anonymizer with per-entity policies. [presidio.dataprivacystack](https://presidio.dataprivacystack.org/getting_started/getting_started_text/)

## 1. Create an environment

Use a virtual environment so dependencies and models stay isolated:

```bash
python -m venv .venv
source .venv/bin/activate   # macOS/Linux
# .venv\Scripts\activate    # Windows PowerShell
```

Install Presidio and spaCy’s English model:

```bash
pip install presidio-analyzer presidio-anonymizer
python -m spacy download en_core_web_lg
```

Presidio’s default NLP engine is spaCy; `en_core_web_lg` is the recommended model for the standard text-de-identification flow. [presidio.dataprivacystack](https://presidio.dataprivacystack.org/getting_started/getting_started_text/)

## 2. Basic detect-and-mask script

Create `pii_masker.py`:

```python
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

text = """
Contact Priya Sharma at priya.sharma@example.com or +91 98765 43210.
Her card number is 4111 1111 1111 1111.
She lives in Bengaluru, India.
"""

# 1. Detect PII
analyzer = AnalyzerEngine()
results = analyzer.analyze(
    text=text,
    language="en",
    entities=[
        "PERSON",
        "EMAIL_ADDRESS",
        "PHONE_NUMBER",
        "CREDIT_CARD",
        "LOCATION",
    ],
)

# 2. Apply a policy per entity type
anonymizer = AnonymizerEngine()

anonymized = anonymizer.anonymize(
    text=text,
    analyzer_results=results,
    operators={
        "PERSON": OperatorConfig("replace", {"new_value": "[PERSON]"}),
        "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "[EMAIL]"}),
        "PHONE_NUMBER": OperatorConfig(
            "mask",
            {
                "masking_char": "*",
                "chars_to_mask": 8,
                "from_end": True,
            },
        ),
        "CREDIT_CARD": OperatorConfig("redact", {}),
        "LOCATION": OperatorConfig("replace", {"new_value": "[LOCATION]"}),
        "DEFAULT": OperatorConfig("redact", {}),
    },
)

print(anonymized.text)
```

The analyzer returns detected PII spans; the anonymizer then applies the configured operator to each span. Presidio supports `replace`, `redact`, `mask`, `hash`, and `encrypt`. [presidio.dataprivacystack](https://presidio.dataprivacystack.org/getting_started/getting_started_text/)

## 3. Example output

Exact output can vary with model and recognizer versions, but it will resemble:

```text
Contact [PERSON] at [EMAIL] or +91 ********3210.
Her card number is <CREDIT_CARD>.
She lives in [LOCATION], India.
```

Use `redact` for high-risk values such as full card numbers and government IDs. Do not rely on partial masking for those values unless your security and compliance teams explicitly approve it.

## 4. Add custom internal identifiers

For an internal service such as **AI-1**, add recognizers for company-specific IDs, for example employee IDs:

```python
from presidio_analyzer import AnalyzerEngine, PatternRecognizer

employee_recognizer = PatternRecognizer(
    supported_entity="EMPLOYEE_ID",
    patterns=[
        {
            "name": "employee id",
            "regex": r"\bEMP-\d{6}\b",
            "score": 0.95,
        }
    ],
    context=["employee", "staff", "ID"],
)

analyzer = AnalyzerEngine()
analyzer.registry.add_recognizer(employee_recognizer)

text = "Employee EMP-123456 requested access."
results = analyzer.analyze(text=text, language="en")

anonymizer = AnonymizerEngine()
result = anonymizer.anonymize(
    text=text,
    analyzer_results=results,
    operators={
        "EMPLOYEE_ID": OperatorConfig("replace", {"new_value": "[EMPLOYEE_ID]"}),
    },
)

print(result.text)
```

Custom `PatternRecognizer` rules are appropriate for deterministic identifiers such as `EMP-123456`, account numbers, MRNs, or internal ticket IDs. Presidio lets you add custom recognizers to its analyzer registry.

## 5. Reversible de-identification

If your workflow needs to restore the original value later—for example, after authorized human review—use **encryption** rather than plain masking:

```python
from cryptography.fernet import Fernet
from presidio_anonymizer.entities import OperatorConfig

key = Fernet.generate_key()
fernet = Fernet(key)

encrypted = fernet.encrypt(b"priya.sharma@example.com").decode()

result = anonymizer.anonymize(
    text=text,
    analyzer_results=results,
    operators={
        "EMAIL_ADDRESS": OperatorConfig(
            "encrypt",
            {"key": key.decode()},
        ),
    },
)
```

Presidio’s `encrypt` operator replaces the original PII with encrypted text using a supplied cryptographic key.  Store the key in a KMS, secrets manager, or HSM—not in source control, environment files committed to Git, or application logs. [microsoft.github](https://microsoft.github.io/presidio/anonymizer/index.html)

## 6. Protect LLM prompts and responses

For an AI service, scan both directions:

```python
def sanitize_for_llm(text: str) -> str:
    results = analyzer.analyze(
        text=text,
        language="en",
        return_decision_process=False,
    )

    return anonymizer.anonymize(
        text=text,
        analyzer_results=results,
        operators={
            "DEFAULT": OperatorConfig("replace", {"new_value": "[REDACTED]"}),
        },
    ).text


def inspect_model_output(text: str) -> str:
    results = analyzer.analyze(text=text, language="en")

    return anonymizer.anonymize(
        text=text,
        analyzer_results=results,
        operators={
            "DEFAULT": OperatorConfig("redact", {}),
        },
    ).text
```

Apply `sanitize_for_llm()` before calling the model and `inspect_model_output()` before persisting or returning the response. This helps prevent PII leakage through prompts, logs, cached context, and generated replies.

## 7. Run it as an API

For a service boundary, wrap the pipeline in FastAPI:

```python
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

class PIIRequest(BaseModel):
    text: str
    language: str = "en"

@app.post("/sanitize")
def sanitize(request: PIIRequest):
    results = analyzer.analyze(
        text=request.text,
        language=request.language,
    )

    anonymized = anonymizer.anonymize(
        text=request.text,
        analyzer_results=results,
        operators={
            "DEFAULT": OperatorConfig("replace", {"new_value": "[REDACTED]"}),
        },
    )

    return {
        "sanitized_text": anonymized.text,
        "detected_entities": [
            {
                "entity_type": r.entity_type,
                "start": r.start,
                "end": r.end,
                "score": r.score,
            }
            for r in results
        ],
    }
```

Run it with:

```bash
uvicorn pii_masker:app --host 0.0.0.0 --port 8000
```

Presidio also offers separate analyzer and anonymizer Docker containers, which can be useful if you want PII handling isolated from application services. [presidio.dataprivacystack](https://presidio.dataprivacystack.org/getting_started/getting_started_text/)

## 8. Operational safeguards

- Log entity type, confidence, action, policy version, and request ID—but never raw PII or the original prompt.
- Use `redact` by default for logs, exports, and third-party LLM calls.
- Use `encrypt` or vault-based tokenization only when authorized reversal is required.
- Add custom recognizers for India-specific or organization-specific identifiers, such as Aadhaar, PAN, employee IDs, and internal account numbers.
- Test recall with real-world samples, multilingual text, OCR output, tables, and adversarial formatting.
- Keep model files, encryption keys, and Presidio configuration versioned and access-controlled.
- Remember that no detector catches everything; Presidio should be one layer in a broader data-minimization, access-control, encryption, logging, and retention strategy. [presidio.dataprivacystack](https://presidio.dataprivacystack.org/getting_started/getting_started_text/)